<a href="https://colab.research.google.com/github/Kira-Stargazer/Aquadex-AI/blob/main/Aquadex_AI_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


In [49]:
import sys
import numpy as np
import pandas as pd
import streamlit as st
import torch
import ultralytics

print("Python     :", sys.version)
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("Streamlit  :", st.__version__)
print("Ultralytics:", ultralytics.__version__)
print("PyTorch    :", torch.__version__)
print("CUDA       :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU")

Python     : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
NumPy      : 2.1.3
Pandas     : 2.2.3
Streamlit  : 1.62.0
Ultralytics: 8.4.127
PyTorch    : 2.11.0+cpu
CUDA       : False


In [50]:
import os
import glob

MODEL_SEARCH_ROOT = "/content/drive/MyDrive/Marine_Debris_Project"

model_files = glob.glob(
    MODEL_SEARCH_ROOT + "/**/*.pt",
    recursive=True
)

print("Searching for YOLO model files...\n")

for model in model_files:
    print(model)

best_models = [
    m for m in model_files
    if os.path.basename(m) == "best.pt"
]

print("\nBest models found:")

for m in best_models:
    print(m)

Searching for YOLO model files...

/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v1/weights/best.pt
/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v1/weights/last.pt
/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/last.pt
/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/best.pt

Best models found:
/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v1/weights/best.pt
/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/best.pt


In [51]:
MODEL_PATH = (
    "/content/drive/MyDrive/"
    "Marine_Debris_Project/training/"
    "sss_shipwreck_v2_1024/weights/best.pt"
)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}"
    )

print("Using model:")
print(MODEL_PATH)

Using model:
/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/best.pt


In [52]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

print("YOLO model loaded successfully.")
print("Model:", MODEL_PATH)

YOLO model loaded successfully.
Model: /content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/best.pt


In [53]:
BASE_DIR = "/content/drive/MyDrive/Marine_Debris_Project"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "predictions",
    "aqueadex_ai"
)

IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")
JSON_DIR = os.path.join(OUTPUT_DIR, "json")
CSV_DIR = os.path.join(OUTPUT_DIR, "csv")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Output directories created:")
print(IMAGE_DIR)
print(JSON_DIR)
print(CSV_DIR)

Output directories created:
/content/drive/MyDrive/Marine_Debris_Project/predictions/aqueadex_ai/images
/content/drive/MyDrive/Marine_Debris_Project/predictions/aqueadex_ai/json
/content/drive/MyDrive/Marine_Debris_Project/predictions/aqueadex_ai/csv


In [54]:
%%writefile /content/app.py

import os
import io
import json
import base64
from datetime import datetime

import streamlit as st
import pandas as pd
import numpy as np
from PIL import Image

from ultralytics import YOLO


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = (
    "/content/drive/MyDrive/"
    "Marine_Debris_Project/training/"
    "sss_shipwreck_v2_1024/weights/best.pt"
)

BASE_DIR = "/content/drive/MyDrive/Marine_Debris_Project"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "predictions",
    "aqueadex_ai"
)

IMAGE_DIR = os.path.join(
    OUTPUT_DIR,
    "images"
)

JSON_DIR = os.path.join(
    OUTPUT_DIR,
    "json"
)

CSV_DIR = os.path.join(
    OUTPUT_DIR,
    "csv"
)

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Aquandex AI",
    page_icon="🌊",
    layout="wide"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    """
    <style>

    .main-title {
        font-size: 42px;
        font-weight: 800;
        margin-bottom: 5px;
    }

    .subtitle {
        font-size: 18px;
        opacity: 0.75;
        margin-bottom: 25px;
    }

    .status-box {
        padding: 18px;
        border-radius: 12px;
        text-align: center;
        font-size: 24px;
        font-weight: 800;
        margin-top: 20px;
        margin-bottom: 25px;
    }

    .detected {
        background-color: #0b4d27;
        color: white;
    }

    .not-detected {
        background-color: #5a1c1c;
        color: white;
    }

    .gps-box {
        padding: 18px;
        border-radius: 12px;
        border: 1px solid rgba(128,128,128,0.3);
        margin-top: 10px;
        margin-bottom: 20px;
    }

    .gps-value {
        font-size: 22px;
        font-weight: 700;
    }

    .warning-box {
        padding: 18px;
        border-radius: 10px;
        background-color: #4b4a12;
        color: white;
        margin-top: 20px;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# LOAD MODEL
# ============================================================

@st.cache_resource
def load_model():

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"YOLO model not found:\n{MODEL_PATH}"
        )

    return YOLO(MODEL_PATH)


model = load_model()


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("⚙️ Analysis Settings")

confidence_threshold = st.sidebar.slider(
    "Confidence threshold",
    min_value=0.10,
    max_value=0.95,
    value=0.70,
    step=0.05
)

image_size = st.sidebar.selectbox(
    "Image size",
    [640, 768, 1024],
    index=2
)

st.sidebar.divider()

st.sidebar.title("🧠 AI Model")

st.sidebar.write("YOLO11n-Seg")
st.sidebar.write("Shipwreck / anomaly segmentation")
st.sidebar.write("Model: v2_1024")

st.sidebar.divider()

st.sidebar.title("📍 GPS")

gps_latitude = st.sidebar.number_input(
    "Latitude",
    min_value=-90.0,
    max_value=90.0,
    value=0.0,
    step=0.000001,
    format="%.6f"
)

gps_longitude = st.sidebar.number_input(
    "Longitude",
    min_value=-180.0,
    max_value=180.0,
    value=0.0,
    step=0.000001,
    format="%.6f"
)

gps_source = st.sidebar.selectbox(
    "GPS source",
    [
        "Manual GPS input",
        "Sonar metadata",
        "External navigation system"
    ]
)


# ============================================================
# HEADER
# ============================================================

st.markdown(
    '<div class="main-title">🌊 Aquadex AI</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'AI-Powered Automated Underwater Marine Debris and '
    'Anomaly Detection using Side-Scan Sonar Imagery'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# GPS DISPLAY
# ============================================================

st.subheader("📍 GPS Location")

gps_col1, gps_col2, gps_col3 = st.columns(3)

with gps_col1:
    st.metric(
        "Latitude",
        f"{gps_latitude:.6f}"
    )

with gps_col2:
    st.metric(
        "Longitude",
        f"{gps_longitude:.6f}"
    )

with gps_col3:
    st.metric(
        "GPS Source",
        gps_source
    )

if gps_latitude != 0.0 or gps_longitude != 0.0:

    google_maps_url = (
        "https://www.google.com/maps/search/?api=1"
        f"&query={gps_latitude},{gps_longitude}"
    )

    st.markdown(
        f"""
        <div class="gps-box">

        <div class="gps-value">
        📍 {gps_latitude:.6f}, {gps_longitude:.6f}
        </div>

        <br>

        <a href="{google_maps_url}" target="_blank">
        🌍 Open GPS Location in Google Maps
        </a>

        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# IMAGE UPLOAD
# ============================================================

st.subheader("📷 Input Sonar Image")

uploaded_file = st.file_uploader(
    "Upload a side-scan sonar image",
    type=[
        "png",
        "jpg",
        "jpeg",
        "tif",
        "tiff"
    ]
)


# ============================================================
# ANALYSIS
# ============================================================

if uploaded_file is not None:

    image_bytes = uploaded_file.read()

    input_image = Image.open(
        io.BytesIO(image_bytes)
    ).convert("RGB")

    st.subheader("🔍 Sonar Image")

    col1, col2 = st.columns(2)

    with col1:

        st.image(
            input_image,
            caption="Input Sonar Image",
            use_container_width=True
        )

    analyze = st.button(
        "🌊 ANALYZE WITH AQUADEX AI",
        type="primary",
        use_container_width=True
    )

    if analyze:

        with st.spinner(
            "Running YOLO sonar analysis..."
        ):

            image_array = np.array(input_image)

            results = model.predict(
                source=image_array,
                conf=confidence_threshold,
                imgsz=image_size,
                verbose=False
            )

            result = results[0]

            # ------------------------------------------------
            # Detection information
            # ------------------------------------------------

            detections = []

            names = result.names

            if result.boxes is not None:

                boxes = result.boxes

                for i in range(len(boxes)):

                    confidence = float(
                        boxes.conf[i].item()
                    )

                    class_id = int(
                        boxes.cls[i].item()
                    )

                    class_name = names.get(
                        class_id,
                        str(class_id)
                    )

                    xyxy = boxes.xyxy[i].cpu().numpy()

                    x1, y1, x2, y2 = [
                        float(v) for v in xyxy
                    ]

                    detection = {
                        "detection_id": i + 1,
                        "class_id": class_id,
                        "class_name": class_name,
                        "confidence": round(
                            confidence,
                            4
                        ),
                        "x1": round(x1, 2),
                        "y1": round(y1, 2),
                        "x2": round(x2, 2),
                        "y2": round(y2, 2),
                        "latitude": gps_latitude,
                        "longitude": gps_longitude
                    }

                    detections.append(
                        detection
                    )

            # ------------------------------------------------
            # Annotated image
            # ------------------------------------------------

            annotated_array = result.plot()

            annotated_image = Image.fromarray(
                annotated_array[..., ::-1]
            )

            # ------------------------------------------------
            # File name
            # ------------------------------------------------

            original_name = os.path.splitext(
                uploaded_file.name
            )[0]

            timestamp = datetime.now().strftime(
                "%Y%m%d_%H%M%S"
            )

            base_filename = (
                f"{original_name}_aquadex_{timestamp}"
            )

            image_path = os.path.join(
                IMAGE_DIR,
                base_filename + ".png"
            )

            json_path = os.path.join(
                JSON_DIR,
                base_filename + ".json"
            )

            csv_path = os.path.join(
                CSV_DIR,
                base_filename + ".csv"
            )

            # ------------------------------------------------
            # Save annotated image
            # ------------------------------------------------

            annotated_image.save(
                image_path
            )

            # ------------------------------------------------
            # Create report
            # ------------------------------------------------

            highest_confidence = 0.0

            if detections:

                highest_confidence = max(
                    d["confidence"]
                    for d in detections
                )

            report = {

                "application":
                    "Aquadex AI",

                "project":
                    "AI-Powered Automated Underwater "
                    "Marine Debris and Anomaly Detection "
                    "System using Side-Scan Sonar Imagery",

                "model":
                    "YOLO11n-Seg",

                "model_version":
                    "sss_shipwreck_v2_1024",

                "task":
                    "Detection + Segmentation",

                "image":
                    uploaded_file.name,

                "image_size":
                    image_size,

                "confidence_threshold":
                    confidence_threshold,

                "timestamp":
                    timestamp,

                "gps": {

                    "latitude":
                        gps_latitude,

                    "longitude":
                        gps_longitude,

                    "source":
                        gps_source
                },

                "summary": {

                    "detections":
                        len(detections),

                    "highest_confidence":
                        highest_confidence,

                    "candidate_detected":
                        len(detections) > 0
                },

                "detections":
                    detections,

                "output_files": {

                    "annotated_image":
                        image_path,

                    "json":
                        json_path,

                    "csv":
                        csv_path
                }
            }

            # ------------------------------------------------
            # Save JSON
            # ------------------------------------------------

            with open(
                json_path,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    report,
                    f,
                    indent=4
                )

            # ------------------------------------------------
            # Save CSV
            # ------------------------------------------------

            if detections:

                csv_df = pd.DataFrame(
                    detections
                )

            else:

                csv_df = pd.DataFrame(
                    [{
                        "detection_id": None,
                        "class_id": None,
                        "class_name": "No detection",
                        "confidence": None,
                        "x1": None,
                        "y1": None,
                        "x2": None,
                        "y2": None,
                        "latitude": gps_latitude,
                        "longitude": gps_longitude
                    }]
                )

            csv_df.to_csv(
                csv_path,
                index=False
            )

            # ------------------------------------------------
            # Save session data
            # ------------------------------------------------

            st.session_state["report"] = report
            st.session_state["annotated_image"] = (
                annotated_image
            )

            st.session_state["image_path"] = (
                image_path
            )

            st.session_state["json_path"] = (
                json_path
            )

            st.session_state["csv_path"] = (
                csv_path
            )

            st.session_state["detections"] = (
                detections
            )

            st.success(
                "Analysis completed successfully."
            )


# ============================================================
# DISPLAY RESULTS
# ============================================================

if "report" in st.session_state:

    report = st.session_state["report"]

    detections = st.session_state[
        "detections"
    ]

    annotated_image = st.session_state[
        "annotated_image"
    ]

    st.divider()

    # --------------------------------------------------------
    # Status
    # --------------------------------------------------------

    if len(detections) > 0:

        st.markdown(
            '<div class="status-box detected">'
            '🟢 SHIPWRECK / ANOMALY CANDIDATE DETECTED'
            '</div>',
            unsafe_allow_html=True
        )

    else:

        st.markdown(
            '<div class="status-box not-detected">'
            '🔴 NO CANDIDATE DETECTED'
            '</div>',
            unsafe_allow_html=True
        )

    # --------------------------------------------------------
    # Images
    # --------------------------------------------------------

    st.subheader("🧠 Aquadex AI Detection")

    st.image(
        annotated_image,
        caption="YOLO11n-Seg Annotated Detection",
        use_container_width=True
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    st.subheader("📊 Analysis Summary")

    metric1, metric2, metric3 = st.columns(3)

    with metric1:

        st.metric(
            "Candidates",
            len(detections)
        )

    with metric2:

        st.metric(
            "Highest Confidence",
            f"{report['summary']['highest_confidence'] * 100:.1f}%"
        )

    with metric3:

        st.metric(
            "Threshold",
            f"{report['confidence_threshold'] * 100:.0f}%"
        )

    # --------------------------------------------------------
    # GPS result
    # --------------------------------------------------------

    st.subheader("📍 GPS Output")

    gps1, gps2 = st.columns(2)

    with gps1:

        st.metric(
            "Latitude",
            f"{report['gps']['latitude']:.6f}"
        )

    with gps2:

        st.metric(
            "Longitude",
            f"{report['gps']['longitude']:.6f}"
        )

    if (
        report["gps"]["latitude"] != 0.0
        or
        report["gps"]["longitude"] != 0.0
    ):

        maps_url = (
            "https://www.google.com/maps/search/?api=1"
            f"&query="
            f"{report['gps']['latitude']},"
            f"{report['gps']['longitude']}"
        )

        st.markdown(
            f"""
            ### 🌍 Location

            **GPS:**
            `{report['gps']['latitude']:.6f}, `
            `{report['gps']['longitude']:.6f}`

            **Source:**
            `{report['gps']['source']}`

            [📍 Open Location in Google Maps]({maps_url})
            """
        )

    # --------------------------------------------------------
    # Individual detections
    # --------------------------------------------------------

    st.subheader("🔎 Individual Detections")

    if detections:

        detection_df = pd.DataFrame(
            detections
        )

        st.dataframe(
            detection_df,
            use_container_width=True,
            hide_index=True
        )

    else:

        st.info(
            "No detections above the selected "
            "confidence threshold."
        )

    # --------------------------------------------------------
    # Export
    # --------------------------------------------------------

    st.subheader("📥 Export Reports")

    json_bytes = json.dumps(
        report,
        indent=4
    ).encode("utf-8")

    csv_bytes = csv_df = pd.DataFrame(
        detections
    ).to_csv(
        index=False
    ).encode("utf-8") if detections else (
        pd.DataFrame([{
            "latitude":
                report["gps"]["latitude"],
            "longitude":
                report["gps"]["longitude"],
            "result":
                "No detection"
        }]).to_csv(
            index=False
        ).encode("utf-8")
    )

    image_buffer = io.BytesIO()

    annotated_image.save(
        image_buffer,
        format="PNG"
    )

    image_bytes = image_buffer.getvalue()

    download1, download2, download3 = st.columns(3)

    with download1:

        st.download_button(
            "📄 Download JSON",
            data=json_bytes,
            file_name=os.path.basename(
                st.session_state["json_path"]
            ),
            mime="application/json",
            use_container_width=True
        )

    with download2:

        st.download_button(
            "📊 Download CSV",
            data=csv_bytes,
            file_name=os.path.basename(
                st.session_state["csv_path"]
            ),
            mime="text/csv",
            use_container_width=True
        )

    with download3:

        st.download_button(
            "🖼️ Download Image",
            data=image_bytes,
            file_name=os.path.basename(
                st.session_state["image_path"]
            ),
            mime="image/png",
            use_container_width=True
        )

    # --------------------------------------------------------
    # Saved files
    # --------------------------------------------------------

    st.subheader("💾 Saved to Google Drive")

    st.code(
        f"""
Annotated image:
{st.session_state["image_path"]}

JSON report:
{st.session_state["json_path"]}

CSV report:
{st.session_state["csv_path"]}
        """,
        language="text"
    )

    # --------------------------------------------------------
    # AI Interpretation
    # --------------------------------------------------------

    st.subheader("🧠 AI Interpretation")

    if detections:

        st.write(
            f"Aquadex AI identified "
            f"**{len(detections)} potential "
            f"shipwreck/anomaly region(s)** "
            f"above the selected confidence threshold."
        )

        st.write(
            f"The highest-confidence candidate "
            f"has a confidence of "
            f"**{report['summary']['highest_confidence'] * 100:.1f}%**."
        )

    else:

        st.write(
            "Aquadex AI did not identify a candidate "
            "above the selected confidence threshold."
        )

    st.markdown(
        """
        <div class="warning-box">

        ⚠️ <b>Research prototype:</b>
        An AI detection represents a candidate region
        and is not confirmation of a shipwreck,
        archaeological site, or marine object.
        Human verification is required.

        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "🌊 Aquadex AI • AI-assisted sonar analysis "
    "for underwater marine exploration • Research Prototype"
)

Overwriting /content/app.py


In [55]:
import os

print("Streamlit application created:")
print(os.path.exists("/content/app.py"))

print("\nFile:")
print("/content/app.py")

Streamlit application created:
True

File:
/content/app.py


In [56]:
import subprocess
import time
import os

# Stop any old Streamlit process
os.system("pkill -f 'streamlit run' || true")

time.sleep(2)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Streamlit started on port 8501.")

time.sleep(5)

print("Checking Streamlit...")

Streamlit started on port 8501.
Checking Streamlit...


In [57]:
import os
import subprocess

cloudflared_path = "/content/cloudflared"

if not os.path.exists(cloudflared_path):

    print("Downloading cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            cloudflared_path
        ],
        check=True
    )

    os.chmod(
        cloudflared_path,
        0o755
    )

print("cloudflared ready.")

version = subprocess.run(
    [cloudflared_path, "--version"],
    capture_output=True,
    text=True
)

print(version.stdout)

cloudflared ready.
cloudflared version 2026.8.2 (built 2026-08-14-12:17 UTC)



In [58]:
import subprocess
import re
import time
import os

cloudflared_path = "/content/cloudflared"

# Kill old tunnel if one exists
os.system("pkill -f cloudflared || true")

time.sleep(2)

tunnel_process = subprocess.Popen(
    [
        cloudflared_path,
        "tunnel",
        "--url",
        "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None

start_time = time.time()

while time.time() - start_time < 30:

    line = tunnel_process.stdout.readline()

    if line:

        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:

            public_url = match.group(0)

            break

    else:

        time.sleep(0.2)


if public_url:

    print("\n" + "=" * 70)
    print("🌊 AQUADEX AI PUBLIC URL")
    print("=" * 70)
    print(public_url)
    print("=" * 70)

else:

    print("\nCould not automatically detect the URL.")
    print("The tunnel process may still be starting.")

2026-08-24T11:44:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-24T11:44:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-24T11:44:25Z INF +--------------------------------------------------------------------------------------------+
2026-08-24T11:44:25Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-24T11:44:25Z INF |  https://invention-sims-rapids-demonstrate.trycloudfla

In [59]:
from IPython.display import display, HTML

if public_url:

    display(
        HTML(
            f"""
            <div style="
                padding:20px;
                border-radius:12px;
                background:#0b4d27;
                color:white;
                font-size:20px;
                margin-top:20px;
            ">
                🌊 <b>Aquadex AI is running</b><br><br>

                <a href="{public_url}"
                   target="_blank"
                   style="color:white;font-size:24px;">
                   🚀 Open Aquadex AI Dashboard
                </a>

                <br><br>

                <code style="color:white;">
                {public_url}
                </code>
            </div>
            """
        )
    )

In [60]:
import requests

try:

    response = requests.get(
        "http://127.0.0.1:8501",
        timeout=10
    )

    print("Streamlit status:", response.status_code)

    if response.status_code == 200:
        print("✅ Streamlit is running correctly.")

except Exception as e:

    print("❌ Streamlit check failed:")
    print(e)

Streamlit status: 200
✅ Streamlit is running correctly.
